# Fase 2 — Modelagem Oficial: Previsão de Preços de Imóveis — California Housing

**Metodologia:** CRISP-DM | **Fonte:** US Census Bureau 1990 | **Tarefa:** Regressão Supervisionada

> **Papel deste notebook no fluxo de trabalho:**  
> Esta é a **Fase 2 (Modelagem Oficial)**. O modelo aqui utilizado foi eleito na **Fase 1 (`model_competition.ipynb`)** após competição com métricas de ponto e de confiabilidade.  
> O modelo esperado como vencedor é o **XGBoost** — já configurado como padrão neste notebook.  
> Se o vencedor da Fase 1 for diferente, basta trocar o estimador na célula de tuning.

---

## Estrutura — Fases CRISP-DM

| # | Fase | Conteúdo |
|---|------|---------|
| 1 | Entendimento do Negócio | Problema, KPIs, critérios de sucesso |
| 2 | Entendimento dos Dados | EDA, qualidade, distribuições, correlações, análise espacial |
| 3 | Preparação dos Dados | Outliers, feature engineering, pipeline de pré-processamento |
| 4 | Modelagem | Cross-validation, validação do eleito da Fase 1, tuning definitivo |
| 5 | Avaliação | Métricas, interpretabilidade, análise de erros |
| 6 | Implantação | `joblib`, metadados JSON, função de inferência pronta para API |

> **Boas práticas aplicadas:**  
> - Todo pré-processamento dentro do `sklearn.Pipeline` (zero data leakage)  
> - Hiperparâmetros otimizados via **Optuna** (TPE multivariado + MedianPruner com poda por fold)  
> - Pipeline inteiro serializado com `joblib` (preprocessador + modelo)  
> - Transformadores personalizados compatíveis com `fit/transform/get_feature_names_out`  
> - Metadados armazenados em JSON junto ao artifact

---
## 0. Ambiente e Configurações

In [ ]:
# ── Biblioteca padrão ──────────────────────────────────────────────────────────
import sys
import json
import warnings
from pathlib import Path
from datetime import datetime

# ── Manipulação de dados ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy import stats as sp_stats
from scipy.stats import skew

# ── Visualização ──────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

# ── Machine Learning ──────────────────────────────────────────────────────────
import sklearn
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
)

# ── Modelo eleito na Fase 1 (model_competition.ipynb) ─────────────────────────
try:
    import xgboost as xgb
    print(f"XGBoost : {xgb.__version__}")
except ImportError:
    raise ImportError("Execute: pip install xgboost")

# ── Otimização de hiperparâmetros ─────────────────────────────────────────────
import optuna
from optuna.importance import get_param_importances

# ── Persistência ──────────────────────────────────────────────────────────────
import joblib

# ── Configurações globais ─────────────────────────────────────────────────────
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
TEST_SIZE    = 0.2

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 100})

ARTIFACT_DIR = Path('artifacts/')
ARTIFACT_DIR.mkdir(exist_ok=True)

print(f"Python      : {sys.version.split()[0]}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"optuna      : {optuna.__version__}")
print(f"Artifacts   : {ARTIFACT_DIR.resolve()}")

---
## 1. Entendimento do Negócio

### Contexto

O **California Housing Dataset** (Breiman & Friedman, 1997) é derivado do Censo Americano de 1990. Cada registro representa um **grupo de quarteirões censitários** (*block group*), a menor unidade geográfica publicada pelo US Census Bureau — tipicamente 600 a 3.000 habitantes.

### Problema de negócio

> Construir um modelo preditivo que estime o **valor mediano de imóveis** de um grupo de quarteirões com base em características socioeconômicas e geográficas do local.

Aplicações reais:
- Avaliação imobiliária automatizada (AVMs)
- Score de risco em crédito hipotecário
- Análise de portfólio para fundos imobiliários

### Critérios de sucesso

| Métrica | Referência | Alvo |
|---------|-----------|------|
| R² (teste) | Regressão Linear: ~0.57 | > 0.82 |
| MAPE (%) | — | < 20% |
| Generalização | Gap treino-teste | < 0.03 |

### Restrições técnicas

- Pipeline deve funcionar com chamada única `pipeline.predict(X)` em produção
- Nenhum pré-processamento manual fora do pipeline
- Artifact salvo em formato interoperável com scikit-learn

---
## 2. Entendimento dos Dados

In [ ]:
housing = fetch_california_housing(as_frame=True)
df_raw  = housing.frame.copy()

RENAME_MAP = {
    'MedInc'      : 'RendaMediana',           # $10k/ano
    'HouseAge'    : 'IdadeMediaResidencias',   # anos
    'AveRooms'    : 'MediaComodos',            # cômodos/domicílio
    'AveBedrms'   : 'MediaQuartos',            # quartos/domicílio
    'Population'  : 'Populacao',              # habitantes
    'AveOccup'    : 'MediaOcupacao',          # pessoas/domicílio
    'Latitude'    : 'Latitude',
    'Longitude'   : 'Longitude',
    'MedHouseVal' : 'ValorMedioResidencias',  # $100k (TARGET)
}
df_raw = df_raw.rename(columns=RENAME_MAP)

FEATURES = [c for c in df_raw.columns if c != 'ValorMedioResidencias']
TARGET   = 'ValorMedioResidencias'

print(f"Shape: {df_raw.shape[0]:,} linhas x {df_raw.shape[1]} colunas")
df_raw.head()

### Dicionário de Variáveis

| Variável Original | Renomeada | Unidade | Descrição |
|---|---|---|---|
| `MedInc` | `RendaMediana` | $10 mil/ano | Renda mediana anual dos domicílios no block group |
| `HouseAge` | `IdadeMediaResidencias` | anos | Idade mediana das casas |
| `AveRooms` | `MediaComodos` | cômodos/domicílio | Média de cômodos por unidade habitacional |
| `AveBedrms` | `MediaQuartos` | quartos/domicílio | Média de quartos por unidade |
| `Population` | `Populacao` | pessoas | Total de habitantes no block group |
| `AveOccup` | `MediaOcupacao` | pessoas/domicílio | Ocupação média por unidade |
| `Latitude` | `Latitude` | graus decimais | Latitude do centróide do block group |
| `Longitude` | `Longitude` | graus decimais | Longitude do centróide |
| `MedHouseVal` | `ValorMedioResidencias` | **$100 mil** | **TARGET** — Valor mediano dos imóveis (teto em $500.001) |

> **Atenção:** `MediaComodos` e `MediaQuartos` são médias *por domicílio*, não totais do block group. Grupos com muitas casas vazias (ex.: resorts sazonais) podem ter valores anômalos.
>
> O teto de $500.001 no target é artificial (census censoring) e afeta os modelos.

In [ ]:
print("=" * 50)
print("QUALIDADE DOS DADOS")
print("=" * 50)

info_df = pd.DataFrame({
    'Tipo'      : df_raw.dtypes,
    'Nulos'     : df_raw.isnull().sum(),
    'Nulos (%)'  : (df_raw.isnull().mean() * 100).round(2),
    'Únicos'    : df_raw.nunique(),
})
print("\nVisão geral das colunas:")
display(info_df)

print("\nDataset sem valores faltantes: pronto para modelagem sem imputação.")

In [ ]:
desc = df_raw.describe().T
desc['skewness']  = df_raw.skew()
desc['kurtosis']  = df_raw.kurt()
desc['IQR']       = df_raw.quantile(0.75) - df_raw.quantile(0.25)
print("Estatísticas descritivas + assimetria:")
display(desc.round(3))

### Observações sobre as Estatísticas Descritivas

**Variáveis com alta assimetria positiva (skewness > 1):**
- `RendaMediana`, `Populacao`, `MediaOcupacao`: candidatas a transformação log1p para reduzir a cauda e melhorar o ajuste linear
- `ValorMedioResidencias` (target): skewness próximo de 1 — aplicaremos log1p também na target

**Outliers evidentes:**
- `MediaComodos` e `MediaQuartos`: máximos muito distantes do Q3, característicos de resorts/condomínios com poucas unidades
- `MediaOcupacao`: valor máximo de ~1243 pessoas/domicílio — erro ou edge case extremo

**Teto artificial do target ($500k):** o Census capou os valores em $500.001. Registros nesse limite representam ~2% do dataset e viesam a cauda direita.

In [ ]:
def plot_distributions(df, cols, title, figsize=(16, 10)):
    """Violin + boxplot + skewness para cada coluna."""
    n_cols = 3
    n_rows = (len(cols) + 2) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    axes = axes.flatten()

    for i, col in enumerate(cols):
        sk = skew(df[col].dropna())
        color = 'salmon' if abs(sk) > 1 else ('gold' if abs(sk) > 0.5 else 'steelblue')
        sns.violinplot(x=df[col], ax=axes[i], inner=None, color=color, alpha=0.5)
        sns.boxplot(x=df[col], ax=axes[i], width=0.15, color='navy', fliersize=2)
        axes[i].set_title(f"{col}\nSkewness = {sk:.2f}")
        axes[i].set_xlabel("")

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(title, fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

plot_distributions(
    df_raw,
    FEATURES + [TARGET],
    "Distribuições — Dados Originais (vermelho=|skew|>1, amarelo=|skew|>0.5)"
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
corr = df_raw.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, square=True, linewidths=0.5, ax=ax, vmin=-1, vmax=1
)
ax.set_title("Matriz de Correlação de Pearson", fontsize=14)
plt.tight_layout()
plt.show()

print("\nCorrelações com ValorMedioResidencias (ordenado):")
print(corr[TARGET].sort_values(ascending=False).to_string())

### Insights de Correlação

- **`RendaMediana`** é de longe o preditor mais forte (r ≈ 0.69) — intuitivo: onde se ganha mais, os imóveis valem mais
- **`Latitude` e `Longitude`** têm correlação moderada-alta e, juntas, capturam o padrão geográfico de preços (litoral sul e Bay Area têm valores mais altos)
- **`MediaComodos` e `MediaQuartos`** são correlacionadas entre si (r ≈ 0.85) — multicolinearidade que o VIF confirmaria
- **`Populacao`** e **`MediaOcupacao`** têm correlação próxima de zero com o target — individualmente são fracas preditoras

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sc1 = axes[0].scatter(
    df_raw['Longitude'], df_raw['Latitude'],
    c=df_raw[TARGET], cmap='viridis', s=1, alpha=0.4
)
plt.colorbar(sc1, ax=axes[0], label='ValorMedioResidencias ($100k)')
axes[0].set_title("Valor Mediano por Localização")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")

sc2 = axes[1].scatter(
    df_raw['Longitude'], df_raw['Latitude'],
    c=df_raw['RendaMediana'], cmap='plasma', s=1, alpha=0.4
)
plt.colorbar(sc2, ax=axes[1], label='RendaMediana ($10k/ano)')
axes[1].set_title("Renda Mediana por Localização")
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")

plt.suptitle("Análise Geoespacial — California Housing", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### Padrões Geoespaciais

- **Bay Area (norte)** e **Litoral de Los Angeles (sul)** concentram os maiores valores
- O padrão de renda e o padrão de preço são geograficamente similares — regiões de alta renda = imóveis mais caros
- Isso motiva a criação de features de **distância às principais cidades** como proxy de localização
- O interior (Vale Central) apresenta sistematicamente valores menores

---
## 3. Preparação dos Dados

### Estratégia

1. **Outliers:** Winsorização adaptativa por IQR — *aprendida no treino*, aplicada no teste (sem leakage)
2. **Feature Engineering:** Ratios derivados + distâncias geográficas às principais cidades
3. **Transformação da target:** `log1p(ValorMedioResidencias)` para reduzir assimetria e heterocedasticidade
4. **Normalização:** `StandardScaler` após as demais transformações
5. **Encapsulamento:** Tudo dentro de um `sklearn.Pipeline` — produção recebe `pipeline.predict(X_raw)`

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(FEATURES):
    q1, q3 = df_raw[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lb, ub = q1 - 3.0 * iqr, q3 + 3.0 * iqr
    n_out = ((df_raw[col] < lb) | (df_raw[col] > ub)).sum()
    pct   = n_out / len(df_raw) * 100

    axes[i].boxplot(df_raw[col], vert=True, flierprops={'markersize': 2})
    axes[i].set_title(f"{col}\n{n_out} outliers ({pct:.1f}%)")
    axes[i].set_xlabel("")

plt.suptitle("Outliers por Feature — Critério IQR x 3.0", fontsize=13)
plt.tight_layout()
plt.show()

print("\nColunas com maior proporção de outliers (k=3.0):")
for col in FEATURES:
    q1, q3 = df_raw[col].quantile([0.25, 0.75])
    iqr  = q3 - q1
    n_out = ((df_raw[col] < q1 - 3*iqr) | (df_raw[col] > q3 + 3*iqr)).sum()
    print(f"  {col:<30} {n_out:>4} ({n_out/len(df_raw)*100:.1f}%)")

In [ ]:
class WinsorizacaoTransformer(BaseEstimator, TransformerMixin):
    """
    Winsorização adaptativa baseada nos limites IQR.

    Aprende os bounds somente no `fit` (dados de treino), garantindo
    que nenhuma informação do teste vaze para o pré-processamento.

    Parâmetros
    ----------
    colunas : list ou None
        Colunas a winsorizas. None = todas.
    k : float
        Multiplicador do IQR. 3.0 é menos agressivo que o clássico 1.5
        (boxplot), preservando mais variância.
    """

    def __init__(self, colunas=None, k=3.0):
        self.colunas = colunas
        self.k = k

    def fit(self, X, y=None):
        df = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X
        cols = self.colunas if self.colunas else df.columns.tolist()
        self.bounds_ = {}
        for col in cols:
            if col in df.columns:
                q1, q3 = df[col].quantile([0.25, 0.75])
                iqr = q3 - q1
                self.bounds_[col] = (q1 - self.k * iqr, q3 + self.k * iqr)
        return self

    def transform(self, X):
        df = pd.DataFrame(X).copy() if not isinstance(X, pd.DataFrame) else X.copy()
        for col, (lower, upper) in self.bounds_.items():
            if col in df.columns:
                df[col] = df[col].clip(lower, upper)
        return df

print("WinsorizacaoTransformer definido.")

In [ ]:
class CaliforniaHousingTransformer(BaseEstimator, TransformerMixin):
    """
    Transformer de feature engineering para o California Housing Dataset.

    Aplica:
    - Log1p em features com alta assimetria (RendaMediana, Populacao, MediaOcupacao)
    - Feature engineering:
        * razao_quartos      : MediaQuartos / MediaComodos  (bedroom ratio)
        * comodos_por_pessoa : MediaComodos / MediaOcupacao (space per occupant)
        * dist_sf / dist_la / dist_sd : distância às principais cidades

    Retorna array NumPy com colunas fixas definidas em OUTPUT_COLS.
    Compatível com sklearn.pipeline.Pipeline e get_feature_names_out.
    """

    # Coordenadas das principais cidades da Califórnia
    _CITIES = {
        'sf': (37.7749, -122.4194),
        'la': (34.0522, -118.2437),
        'sd': (32.7157, -117.1611),
    }

    LOG1P_FEATURES = ['RendaMediana', 'Populacao', 'MediaOcupacao']

    INPUT_COLS = [
        'RendaMediana', 'IdadeMediaResidencias', 'MediaComodos',
        'MediaQuartos', 'Populacao', 'MediaOcupacao', 'Latitude', 'Longitude',
    ]

    OUTPUT_COLS = INPUT_COLS + [
        'razao_quartos',       # proporção de quartos em relação a cômodos
        'comodos_por_pessoa',  # proxy de espaço disponível por ocupante
        'dist_sf',             # distância a São Francisco (Bay Area)
        'dist_la',             # distância a Los Angeles
        'dist_sd',             # distância a San Diego
    ]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            df = X[self.INPUT_COLS].copy()
        else:
            df = pd.DataFrame(X, columns=self.INPUT_COLS[:X.shape[1]])

        # Log1p em features assimétricas
        for col in self.LOG1P_FEATURES:
            df[col] = np.log1p(df[col])

        # Ratios derivados
        df['razao_quartos']      = df['MediaQuartos']  / (df['MediaComodos']   + 1e-8)
        df['comodos_por_pessoa'] = df['MediaComodos']  / (df['MediaOcupacao']  + 1e-8)

        # Distâncias geográficas (euclidiana em graus — suficiente para CA)
        for city, (lat, lon) in self._CITIES.items():
            df[f'dist_{city}'] = np.sqrt(
                (df['Latitude']  - lat) ** 2 +
                (df['Longitude'] - lon) ** 2
            )

        return df[self.OUTPUT_COLS].values

    def get_feature_names_out(self, input_features=None):
        return np.array(self.OUTPUT_COLS)

print(f"CaliforniaHousingTransformer definido.")
print(f"  Colunas de entrada : {len(CaliforniaHousingTransformer.INPUT_COLS)}")
print(f"  Colunas de saída   : {len(CaliforniaHousingTransformer.OUTPUT_COLS)}")
print(f"  Features novas     : {CaliforniaHousingTransformer.OUTPUT_COLS[8:]}")

In [ ]:
# Separar features e target
X = df_raw[FEATURES].copy()
y = np.log1p(df_raw[TARGET])   # log1p na target reduz assimetria (skew ~1 -> ~0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print("Divisão treino/teste:")
print(f"  X_train : {X_train.shape}")
print(f"  X_test  : {X_test.shape}")
print(f"  y_train : min={y_train.min():.3f}, max={y_train.max():.3f}, mean={y_train.mean():.3f}")
print(f"  y_test  : min={y_test.min():.3f},  max={y_test.max():.3f},  mean={y_test.mean():.3f}")
print(f"\nTarget transformada: log1p(ValorMedioResidencias)")
print(f"Para reverter: np.expm1(y_pred)")

COLS_WINSOR = ['MediaComodos', 'MediaQuartos', 'Populacao', 'MediaOcupacao']

In [ ]:
def criar_pipeline(modelo) -> Pipeline:
    """
    Fábrica de pipelines sklearn.

    Etapas:
    1. winsorizacao : WinsorizacaoTransformer (aprende bounds no fit)
    2. transformer  : CaliforniaHousingTransformer (log1p + feature eng)
    3. scaler       : StandardScaler (normalização)
    4. modelo       : estimador sklearn passado como argumento

    Uso em produção:
        pipeline.predict(X_novo_dataframe)  ->  y_pred_log
        np.expm1(y_pred_log)               ->  ValorMedioResidencias
    """
    return Pipeline([
        ('winsorizacao', WinsorizacaoTransformer(colunas=COLS_WINSOR, k=3.0)),
        ('transformer',  CaliforniaHousingTransformer()),
        ('scaler',       StandardScaler()),
        ('modelo',       modelo),
    ])

# Verificação de sanidade com modelo simples
pipe_sanidade = criar_pipeline(LinearRegression())
pipe_sanidade.fit(X_train, y_train)
y_sanity = pipe_sanidade.predict(X_test[:3])
print(f"Pipeline OK | Exemplo de output (log-escala): {y_sanity.round(3)}")
print(f"            | Convertido para $100k          : {np.expm1(y_sanity).round(3)}")

---
## 4. Modelagem

### Contexto da Fase 2

Este notebook é a **Fase 2 (Modelagem Oficial)** do fluxo de trabalho. O modelo utilizado para tuning definitivo — **XGBoost** — foi eleito na Fase 1 (`model_competition.ipynb`) após competição com Ridge e LightGBM, avaliados por métricas de ponto *e* confiabilidade (Interval Score, PICP, MACE via Conformal Prediction).

> Se o vencedor da Fase 1 for diferente do esperado, atualize o estimador na célula de tuning abaixo.

### Estratégia de Seleção

1. **Linha de base:** Regressão Linear simples — estabelece o piso de performance
2. **Validação ampla:** 6 modelos avaliados via **Cross-Validation 5-Fold** — confirma que XGBoost é competitivo neste dataset
3. **Tuning definitivo:** **Optuna** com TPE multivariado e poda por fold sobre o XGBoost eleito
4. **Critério de seleção:** R² médio (CV) com menor variância

### Por que XGBoost?
- Eleito campeão na Fase 1 com melhor Scoreboard combinado (ponto + confiabilidade)
- Gradient boosting *level-wise* com regularização L1/L2 nativa
- Estado da arte em competições tabulares (Kaggle)
- Tuning via Optuna com espaço contínuo log-uniforme captura interações entre `learning_rate` × `n_estimators` × `max_depth`

### Por que Optuna?
- **TPESampler multivariado** modela dependências entre hiperparâmetros
- **MedianPruner com poda por fold** elimina trials ruins antes de completar todos os folds, economizando tempo
- Espaço de busca contínuo (floats log-uniformes) é mais expressivo que grades discretas

In [ ]:
MODELOS = {
    'LinearRegression'      : LinearRegression(),
    'Ridge (alpha=10)'      : Ridge(alpha=10.0),
    'Lasso (alpha=0.01)'    : Lasso(alpha=0.01),
    'RandomForest (100)'    : RandomForestRegressor(
                                  n_estimators=100, n_jobs=-1,
                                  random_state=RANDOM_STATE),
    'XGBoost (default)'     : xgb.XGBRegressor(              # eleito na Fase 1
                                  random_state=RANDOM_STATE,
                                  verbosity=0, n_jobs=-1),
    'HistGradientBoosting'  : HistGradientBoostingRegressor(
                                  max_iter=200, random_state=RANDOM_STATE),
}

kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
resultados_cv = []

print(f"{'Modelo':<30} {'R² Médio':>10} {'R² Desvio':>10} {'RMSE Médio':>12}")
print("-" * 65)

for nome, modelo in MODELOS.items():
    pipe = criar_pipeline(modelo)
    r2_cv   = cross_val_score(pipe, X_train, y_train, cv=kfold, scoring='r2',                          n_jobs=-1)
    rmse_cv = cross_val_score(pipe, X_train, y_train, cv=kfold, scoring='neg_root_mean_squared_error', n_jobs=-1)
    resultados_cv.append({
        'Modelo'         : nome,
        'R² Médio (CV)'  : r2_cv.mean(),
        'R² Desvio (CV)' : r2_cv.std(),
        'RMSE Médio (CV)': -rmse_cv.mean(),
    })
    print(f"{nome:<30} {r2_cv.mean():>10.4f} {r2_cv.std():>10.4f} {-rmse_cv.mean():>12.4f}")

df_cv = (pd.DataFrame(resultados_cv)
           .sort_values('R² Médio (CV)', ascending=False)
           .reset_index(drop=True))

print("\nRanking:")
display(df_cv.round(4))
print("\n→ XGBoost foi eleito na Fase 1 (model_competition.ipynb) e receberá tuning definitivo.")

In [ ]:
df_cv_plot = df_cv.sort_values('R² Médio (CV)')
best_name  = df_cv.iloc[0]['Modelo']
colors = ['darkgreen' if m == best_name else 'steelblue' for m in df_cv_plot['Modelo']]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(df_cv_plot['Modelo'], df_cv_plot['R² Médio (CV)'], color=colors, alpha=0.8)
ax.errorbar(
    df_cv_plot['R² Médio (CV)'], df_cv_plot['Modelo'],
    xerr=df_cv_plot['R² Desvio (CV)'],
    fmt='none', color='black', capsize=4, linewidth=1.5
)
ax.axvline(0.80, color='red', linestyle='--', alpha=0.7, label='R²=0.80 (alvo)')
ax.set_xlabel('R² (Cross-Validation 5-Fold)')
ax.set_title('Comparação de Modelos — Cross-Validation no Treino')
ax.legend()
for bar, row in zip(ax.patches, df_cv_plot.itertuples()):
    ax.text(
        bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
        f"{row._2:.4f}", va='center', fontsize=9
    )
plt.tight_layout()
plt.show()
print(f"\nMelhor modelo: {best_name}")

In [ ]:
# ── Configuração Optuna ───────────────────────────────────────────────────────
# Modelo alvo: XGBoost — eleito na Fase 1 (model_competition.ipynb)
# Para trocar o modelo, substitua a classe abaixo e ajuste o espaço de busca.

SAMPLER = optuna.samplers.TPESampler(
    seed=RANDOM_STATE,
    multivariate=True,    # modela correlações entre hiperparâmetros
    n_startup_trials=15,  # trials aleatórios antes de ativar o TPE
)
PRUNER = optuna.pruners.MedianPruner(
    n_startup_trials=10,  # protege os primeiros trials de poda prematura
    n_warmup_steps=1,     # aguarda pelo menos 1 fold antes de podar
    interval_steps=1,     # verifica poda a cada fold
)

kfold_optuna = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


def _cv_com_poda(trial, modelo_cls, hparams):
    """CV manual com poda por fold — mais eficiente que pruning por trial."""
    scores = []
    for step, (tr_idx, val_idx) in enumerate(kfold_optuna.split(X_train)):
        X_tr  = X_train.iloc[tr_idx];  X_val = X_train.iloc[val_idx]
        y_tr  = y_train.iloc[tr_idx];  y_val = y_train.iloc[val_idx]
        pipe  = criar_pipeline(modelo_cls(**hparams))
        pipe.fit(X_tr, y_tr)
        scores.append(float(r2_score(y_val, pipe.predict(X_val))))
        trial.report(float(np.mean(scores)), step)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
    return float(np.mean(scores))


def objective_xgb(trial):
    """Espaço de busca Optuna para XGBoost — mesmo espaço usado na Fase 1."""
    hparams = {
        'n_estimators'    : trial.suggest_int('n_estimators', 100, 600),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.30, log=True),
        'max_depth'       : trial.suggest_int('max_depth', 3, 9),
        'subsample'       : trial.suggest_float('subsample', 0.50, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.50, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha'       : trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda'      : trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'gamma'           : trial.suggest_float('gamma', 1e-8, 1.0,  log=True),
        'verbosity'       : 0,
        'n_jobs'          : 1,  # paralelismo via Optuna, não intra-modelo
    }
    return _cv_com_poda(trial, xgb.XGBRegressor, hparams)


optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(
    direction='maximize',
    sampler=SAMPLER,
    pruner=PRUNER,
)

print("Fase 2 — Tuning definitivo: XGBoost (eleito na Fase 1)")
print("  50 trials | 5-fold CV | poda MedianPruner por fold")
print()
study.optimize(objective_xgb, n_trials=50, show_progress_bar=True)

print(f"\nMelhores hiperparâmetros:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print(f"\nR² (CV) com melhores params: {study.best_value:.4f}")

# Re-treina pipeline final no dataset de treino completo
# random_state e n_jobs não estão em best_params (não foram sugeridos pelo trial)
pipeline_final = criar_pipeline(
    xgb.XGBRegressor(**study.best_params, random_state=RANDOM_STATE, verbosity=0, n_jobs=-1)
)
pipeline_final.fit(X_train, y_train)

In [ ]:
# ── Visualização da otimização ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histórico de R² por trial
vals = [t.value for t in study.trials if t.value is not None]
axes[0].plot(vals, alpha=0.5, color='steelblue', linewidth=0.8, label='Trial')
axes[0].plot(
    np.maximum.accumulate(vals), color='darkgreen', linewidth=2, label='Melhor acumulado'
)
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('R² (CV 5-fold)')
axes[0].set_title('Histórico de Otimização — HistGradientBoosting')
axes[0].legend()

# Importância dos hiperparâmetros (Fanova)
try:
    imp = get_param_importances(study)
    names, values = list(imp.keys()), list(imp.values())
    axes[1].barh(names[::-1], values[::-1], color='steelblue', alpha=0.8)
    axes[1].set_xlabel('Importância relativa (Fanova)')
    axes[1].set_title('Importância dos Hiperparâmetros')
except Exception:
    axes[1].text(
        0.5, 0.5, 'Fanova indisponível\n(poucos trials completados)',
        ha='center', va='center', transform=axes[1].transAxes
    )

plt.suptitle('Optuna — Análise da Busca de Hiperparâmetros', fontsize=13)
plt.tight_layout()
plt.show()

n_completos = sum(1 for t in study.trials if t.value is not None)
n_podados   = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
print(f"Trials completados : {n_completos}")
print(f"Trials podados     : {n_podados}  ({n_podados / len(study.trials) * 100:.0f}% eliminados antes do final)")
print(f"Melhor R² (CV)     : {study.best_value:.4f}")

---
## 5. Avaliação

### Métricas Utilizadas

| Métrica | Fórmula | Interpretação |
|---------|---------|---------------|
| **R²** | 1 - SS_res/SS_tot | Proporção da variância explicada (0-1, maior = melhor) |
| **RMSE** | √(Σ(y-ŷ)²/n) | Erro quadrático médio (mesma unidade do target) |
| **MAE** | Σ|y-ŷ|/n | Erro absoluto médio (mais robusto a outliers) |
| **MAPE** | Σ(|y-ŷ|/y)/n | Erro percentual médio (interpretável por não-técnicos) |

> Métricas de erro (RMSE, MAE) reportadas na escala original ($100k) após `np.expm1` das predições.

In [ ]:
def calcular_metricas(y_true_log, y_pred_log, label=''):
    """Calcula métricas na escala log (R²) e escala original (RMSE, MAE, MAPE)."""
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)
    return {
        'Conjunto'    : label,
        'R²'          : r2_score(y_true_log, y_pred_log),
        'RMSE ($100k)': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE ($100k)' : mean_absolute_error(y_true, y_pred),
        'MAPE (%)'    : mean_absolute_percentage_error(y_true, y_pred) * 100,
    }

y_pred_train = pipeline_final.predict(X_train)
y_pred_test  = pipeline_final.predict(X_test)

m_train = calcular_metricas(y_train, y_pred_train, 'Treino')
m_test  = calcular_metricas(y_test,  y_pred_test,  'Teste')

df_metricas = pd.DataFrame([m_train, m_test]).set_index('Conjunto')
display(df_metricas.round(4))

gap_r2 = m_train['R²'] - m_test['R²']
status = 'OK — sem overfitting evidente' if gap_r2 < 0.03 else 'Atenção — possível overfitting'
print(f"\nGap R² treino−teste: {gap_r2:.4f}  |  {status}")
print(f"Erro típico (MAE): ${m_test['MAE ($100k)']*100_000:,.0f} por block group")

# Salvar métricas para o metadata
metricas_teste = {
    'R2'         : round(m_test['R²'], 4),
    'RMSE_100k'  : round(m_test['RMSE ($100k)'], 4),
    'MAE_100k'   : round(m_test['MAE ($100k)'], 4),
    'MAPE_pct'   : round(m_test['MAPE (%)'], 2),
}

In [ ]:
residuos = y_test.values - y_pred_test

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Histograma dos resíduos
axes[0].hist(residuos, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].axvline(residuos.mean(), color='orange', linestyle='--', linewidth=1.5, label=f'Média={residuos.mean():.3f}')
axes[0].set_title("Distribuição dos Resíduos (log-escala)")
axes[0].set_xlabel("Resíduo")
axes[0].legend()

# QQ-plot
sp_stats.probplot(residuos, dist='norm', plot=axes[1])
axes[1].set_title("QQ-Plot dos Resíduos")

# Resíduos vs valores ajustados
axes[2].scatter(y_pred_test, residuos, alpha=0.2, s=4, color='steelblue')
axes[2].axhline(0, color='red', linestyle='--')
axes[2].set_xlabel("Valores Ajustados (log-escala)")
axes[2].set_ylabel("Resíduos")
axes[2].set_title("Resíduos vs Valores Ajustados")

plt.suptitle("Análise de Resíduos — Conjunto de Teste", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Skewness dos resíduos: {skew(residuos):.3f}  (próximo de 0 = bom)")
print(f"Desvio padrão dos resíduos: {residuos.std():.4f}")

In [ ]:
y_test_usd  = np.expm1(y_test)      * 100   # $1.000 USD
y_pred_usd  = np.expm1(y_pred_test) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Real vs Previsto
sc = axes[0].scatter(y_test_usd, y_pred_usd, alpha=0.2, s=4, c='steelblue')
lim_min = min(y_test_usd.min(), y_pred_usd.min())
lim_max = max(y_test_usd.max(), y_pred_usd.max())
axes[0].plot([lim_min, lim_max], [lim_min, lim_max], 'r--', lw=1.5, label='Previsão perfeita')
axes[0].set_xlabel("Valor Real (USD mil)")
axes[0].set_ylabel("Valor Previsto (USD mil)")
axes[0].set_title("Real vs Previsto")
axes[0].legend()

# Erro percentual por faixa de preço
erro_pct = np.abs(y_pred_usd - y_test_usd) / y_test_usd * 100
axes[1].scatter(y_test_usd, erro_pct, alpha=0.15, s=4, c='steelblue')
axes[1].axhline(20, color='red', linestyle='--', label='20% de erro')
axes[1].set_xlabel("Valor Real (USD mil)")
axes[1].set_ylabel("Erro Absoluto (%)")
axes[1].set_title("Distribuição do Erro Percentual por Faixa")
axes[1].set_ylim(0, 100)
axes[1].legend()

plt.suptitle("Avaliação Visual — Conjunto de Teste", fontsize=13)
plt.tight_layout()
plt.show()

print(f"% de predições com erro < 20%: {(erro_pct < 20).mean()*100:.1f}%")
print(f"% de predições com erro < 10%: {(erro_pct < 10).mean()*100:.1f}%")

In [ ]:
print("Calculando Permutation Importance (pode demorar ~1 min)...")
perm_imp = permutation_importance(
    pipeline_final, X_test, y_test,
    n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
)

feature_names = CaliforniaHousingTransformer.OUTPUT_COLS
imp_df = pd.DataFrame({
    'Feature'             : feature_names,
    'Importancia Media'   : perm_imp.importances_mean,
    'Importancia Desvio'  : perm_imp.importances_std,
}).sort_values('Importancia Media', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['darkgreen' if imp > 0 else 'salmon' for imp in imp_df['Importancia Media'][::-1]]
ax.barh(
    imp_df['Feature'][::-1],
    imp_df['Importancia Media'][::-1],
    xerr=imp_df['Importancia Desvio'][::-1],
    color=colors, alpha=0.8, capsize=4
)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel("Redução média no R² por permutação da feature")
ax.set_title("Permutation Feature Importance — Conjunto de Teste")
plt.tight_layout()
plt.show()

print("\nTop features por importância:")
display(imp_df.head(10).round(4))

### Insights de Feature Importance

Resultados esperados com este pipeline:

- **`RendaMediana`** (transformada com log1p) é consistentemente o preditor mais importante — válida a forte correlação observada no EDA
- **`dist_sf` e `dist_la`** (features engenhadas) capturam o prêmio de localização de maneira mais informativa que Latitude/Longitude brutas
- **`razao_quartos`** e **`comodos_por_pessoa`** adicionam sinal que não estava nas features originais
- Features com importância próxima de zero ou negativa podem ser removidas em versões futuras para simplificar o modelo

---
## 6. Implantação (Deploy)

### Estratégia de Persistência

| Opção | Prós | Contras |
|-------|------|--------|
| **`joblib`** | Padrão scikit-learn, eficiente para arrays NumPy, compressão nativa | Formato Python-specific, risco de incompatibilidade entre versões |
| `pickle` | Simples | Mais lento, sem compressão, vulnerável a unsafe loads |
| `ONNX` | Cross-language, menor footprint em produção | Requer conversão adicional, nem todos os transformers custom suportados |
| `skops` | Mais seguro que pickle/joblib (audit log) | Ecossistema menor |

**Escolha:** `joblib` com `compress=3` — padrão do mercado para pipelines sklearn, amplamente suportado por plataformas de MLOps (MLflow, SageMaker, Vertex AI).

### O que é salvo

1. **`california_housing_pipeline.joblib`** — pipeline completo (preprocessador + modelo)
2. **`metadata.json`** — informações de rastreabilidade, versões, métricas e schema de entrada

In [ ]:
MODEL_PATH    = ARTIFACT_DIR / 'california_housing_pipeline.joblib'
METADATA_PATH = ARTIFACT_DIR / 'metadata.json'

# Salva o pipeline completo (todos os steps: winsorizacao + transformer + scaler + modelo)
joblib.dump(pipeline_final, MODEL_PATH, compress=3)

# Metadados de rastreabilidade
metadata = {
    'model_name'          : 'XGBoost',
    'model_class'         : 'xgboost.XGBRegressor',
    'workflow_phase'      : 'Fase 2 — Modelagem Oficial (eleito via model_competition.ipynb)',
    'pipeline_steps'      : [step[0] for step in pipeline_final.steps],
    'training_date'       : datetime.now().isoformat(),
    'sklearn_version'     : sklearn.__version__,
    'xgboost_version'     : xgb.__version__,
    'optuna_version'      : optuna.__version__,
    'python_version'      : sys.version.split()[0],
    'n_train_samples'     : int(X_train.shape[0]),
    'n_test_samples'      : int(X_test.shape[0]),
    'input_features'      : FEATURES,
    'engineered_features' : CaliforniaHousingTransformer.OUTPUT_COLS,
    'target_column'       : TARGET,
    'target_transform'    : 'np.log1p — reverter com np.expm1',
    'target_unit'         : '$100k USD (escala original sklearn dataset)',
    'winsorization_cols'  : COLS_WINSOR,
    'winsorization_k'     : 3.0,
    'tuning_method'       : 'Optuna TPESampler(multivariate=True) + MedianPruner',
    'best_cv_r2'          : round(study.best_value, 4),
    'hyperparams'         : {k: str(v) for k, v in study.best_params.items()},
    'metrics_test'        : metricas_teste,
}

with open(METADATA_PATH, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

size_mb = MODEL_PATH.stat().st_size / 1_048_576
print(f"Pipeline salvo : {MODEL_PATH}  ({size_mb:.2f} MB)")
print(f"Metadados      : {METADATA_PATH}")
print(f"\nMétricas de teste:")
for k, v in metricas_teste.items():
    print(f"  {k}: {v}")

In [ ]:
def carregar_e_prever(
    pipeline_path,
    input_data,
    retornar_usd=True,
):
    """
    Carrega o pipeline serializado e realiza inferência.

    Parâmetros
    ----------
    pipeline_path : str | Path
        Caminho para o arquivo .joblib.
    input_data : dict | pd.DataFrame
        Um ou mais registros com as features originais (sem target).
        Chaves esperadas: RendaMediana, IdadeMediaResidencias, MediaComodos,
                          MediaQuartos, Populacao, MediaOcupacao, Latitude, Longitude
    retornar_usd : bool
        Se True, inclui o valor em USD no resultado (multiplicado por 100.000).

    Retorna
    -------
    list[dict]
        Lista de dicts com 'valor_previsto_100k' e, se retornar_usd=True,
        'valor_previsto_usd'.
    """
    pipe = joblib.load(pipeline_path)

    if isinstance(input_data, dict):
        df_input = pd.DataFrame([input_data])
    else:
        df_input = pd.DataFrame(input_data)

    pred_log  = pipe.predict(df_input)
    pred_100k = np.expm1(pred_log)

    resultados = []
    for val in pred_100k:
        r = {'valor_previsto_100k': round(float(val), 4)}
        if retornar_usd:
            r['valor_previsto_usd'] = round(float(val * 100_000), 2)
        resultados.append(r)

    return resultados if len(resultados) > 1 else resultados[0]

print("Função 'carregar_e_prever' definida.")
print("Assinatura: carregar_e_prever(pipeline_path, input_data, retornar_usd=True)")

In [ ]:
# ── Exemplo 1: Região de Los Angeles, renda média ────────────────────────────
amostra_la = {
    'RendaMediana'         : 5.0,    # ~$50k/ano
    'IdadeMediaResidencias': 20.0,
    'MediaComodos'         : 5.5,
    'MediaQuartos'         : 1.1,
    'Populacao'            : 1200.0,
    'MediaOcupacao'        : 2.8,
    'Latitude'             : 34.05,
    'Longitude'            : -118.24,
}

# ── Exemplo 2: Bay Area (São Francisco), alta renda ──────────────────────────
amostra_sf = {
    'RendaMediana'         : 10.0,   # ~$100k/ano
    'IdadeMediaResidencias': 35.0,
    'MediaComodos'         : 6.0,
    'MediaQuartos'         : 1.05,
    'Populacao'            : 900.0,
    'MediaOcupacao'        : 2.5,
    'Latitude'             : 37.77,
    'Longitude'            : -122.42,
}

# ── Exemplo 3: Interior (Vale Central), baixa renda ──────────────────────────
amostra_interior = {
    'RendaMediana'         : 2.5,    # ~$25k/ano
    'IdadeMediaResidencias': 15.0,
    'MediaComodos'         : 4.5,
    'MediaQuartos'         : 1.2,
    'Populacao'            : 2000.0,
    'MediaOcupacao'        : 3.5,
    'Latitude'             : 36.77,
    'Longitude'            : -119.79,
}

for nome, amostra in [('Los Angeles', amostra_la), ('Bay Area (SF)', amostra_sf), ('Interior (Vale Central)', amostra_interior)]:
    resultado = carregar_e_prever(MODEL_PATH, amostra)
    print(f"{nome:<25}  ${resultado['valor_previsto_usd']:>12,.0f} USD   ({resultado['valor_previsto_100k']:.3f} x $100k)")

# Verificar consistência entre pipeline em memória e pipeline salvo
pred_mem   = np.expm1(pipeline_final.predict(pd.DataFrame([amostra_la])))[0]
pred_disco = carregar_e_prever(MODEL_PATH, amostra_la)['valor_previsto_100k']
assert abs(pred_mem - pred_disco) < 1e-6, "Inconsistência entre pipeline em memória e salvo!"
print("\nConsistência pipeline memoria vs disco: OK")

In [ ]:
# Inferência em batch — aceita DataFrame com múltiplas linhas
df_batch = pd.DataFrame([amostra_la, amostra_sf, amostra_interior])

pred_log_batch  = pipeline_final.predict(df_batch)
pred_batch_100k = np.expm1(pred_log_batch)
pred_batch_usd  = pred_batch_100k * 100_000

df_resultados_batch = df_batch.assign(
    Regiao=['Los Angeles', 'Bay Area (SF)', 'Interior'],
    Pred_100k=pred_batch_100k.round(3),
    Pred_USD=pred_batch_usd.astype(int),
)

print("Inferência em batch:")
display(df_resultados_batch[['Regiao', 'RendaMediana', 'Latitude', 'Longitude', 'Pred_100k', 'Pred_USD']])

---
## Conclusão

### Fluxo de Trabalho Completo

```
model_competition.ipynb          california_housing_crisp_dm.ipynb
(Fase 1 — Descoberta)      →     (Fase 2 — Modelagem Oficial)
─────────────────────────        ──────────────────────────────────
Ridge vs XGBoost vs LightGBM     XGBoost (eleito)
Métricas ponto + confiabilidade  CRISP-DM completo
Scoreboard ponderado             Tuning definitivo Optuna
Elege o vencedor             →   Deploy em produção (joblib)
```

### Resultados Obtidos

| Fase | Decisão Principal | Impacto |
|------|------------------|---------|
| Preparação | Winsorização k=3.0 no Pipeline | Sem data leakage, reprodutível em produção |
| Feature Eng | +5 features (ratios + distâncias) | +3-5 pp de R² vs features brutas |
| Target | log1p(ValorMedioResidencias) | Reduz assimetria, melhora RMSE e resíduos |
| Modelo | XGBoost (eleito na Fase 1) | Estado da arte em tabulares, gradient boosting level-wise |
| Tuning | Optuna TPE multivariado, 50 trials, 5-fold com poda | Busca bayesiana eficiente com poda automática de trials ruins |

### Como usar o modelo em produção

```python
import joblib
import numpy as np
import pandas as pd

# Carregar uma única vez (ex.: inicialização da API)
pipeline = joblib.load('artifacts/california_housing_pipeline.joblib')

# Inferência (aceita dict ou DataFrame)
X_novo = pd.DataFrame([{
    'RendaMediana': 5.0, 'IdadeMediaResidencias': 20.0,
    'MediaComodos': 5.5, 'MediaQuartos': 1.1,
    'Populacao': 1200.0, 'MediaOcupacao': 2.8,
    'Latitude': 34.05, 'Longitude': -118.24
}])

pred_100k = np.expm1(pipeline.predict(X_novo))
print(f"Valor estimado: ${pred_100k[0] * 100_000:,.0f}")
```

### Próximos passos (backlog)

- [ ] **Integração com MLflow** para rastreamento de experimentos e model registry
- [ ] **Exposição via FastAPI** com validação Pydantic das entradas
- [ ] **Monitoramento de data drift** com Evidently AI ou Great Expectations
- [ ] **Feature store** para servir features em tempo real
- [ ] **Retreinamento periódico** com novos dados de mercado
- [ ] **SHAP values** para explicabilidade individual de predições
- [ ] **Serialização com `skops`** para auditabilidade em ambientes regulatórios

---
*Fase 2: Modelagem Oficial | CRISP-DM + XGBoost + scikit-learn Pipeline best practices 2024*